# Model Training and Comparison

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('../dataset/heart_disease.csv')

In [3]:
X = df.drop('Target', axis=1)
y = df['Target']

In [4]:
numeric_features = ['Age', 'Resting BP', 'Serum Cholestrol(mg/dl)', 'Max Heart Rate', 'Oldpeak', 'No. of Major Vessels']
categorical_features = ['Sex', 'Chest Pain', 'Resting ECG Result', 'Slope', 'Thal']
boolean_features = ['Fasting Blood Sugar (> 120 mg/dl)', 'Angina (Exercise Induced)']

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer([

    ("num", Pipeline([
        ("imputer", KNNImputer(n_neighbors=10)),
        ("scaler", StandardScaler())
    ]), numeric_features),

    ("bool", "passthrough", boolean_features),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy='most_frequent')),
        ("encoder", OneHotEncoder())
    ]), categorical_features)
])

## Section 1: Baseline Models

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

baseline_models = {
    "Logistic Regression": LogisticRegression(random_state=42, class_weight='balanced'),
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "SVM": SVC(random_state=42, class_weight='balanced'),
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(random_state=42, class_weight='balanced'),
    "XGBoost": XGBClassifier(random_state=42)
}

In [7]:
from sklearn.model_selection import cross_validate

results = []

for name, model in baseline_models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_validate(pipeline, X, y, cv=5,
        scoring=[
            'accuracy',
            'precision',
            'recall',
            'f1'
        ]
    )

    scores = {
        "Model": name,
        "Accuracy": scores['test_accuracy'].mean(),
        "Precision": scores['test_precision'].mean(),
        "Recall": scores['test_recall'].mean(),
        "F1": scores['test_f1'].mean()
    }

    results.append(scores)

baseline_results = pd.DataFrame(results)
baseline_results

,Model,Accuracy,Precision,Recall,F1
0,Logistic Regression,0.828689,0.820293,0.804497,0.808962
1,KNN,0.838525,0.832143,0.811905,0.819858
2,Naive Bayes,0.808907,0.809912,0.760847,0.779162
3,SVM,0.809016,0.791646,0.790476,0.790066
4,Decision Tree,0.743224,0.732244,0.696561,0.707867
5,Random Forest,0.809126,0.792462,0.790741,0.789714
6,XGBoost,0.799290,0.800217,0.761640,0.775161


## Section 2: Hyperparameter Tuning

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

tuned_results = []

params_grid = {
    "Logistic Regression": {
        "model__C": [0.001, 0.01, 0.1, 1, 10, 100]
    },
    "KNN": {
        "model__n_neighbors": [2, 5, 10, 15, 20],
        "model__weights": ["uniform", "distance"]  
    },
    "SVM": {
        "model__C": [0.01, 0.1, 1, 10, 100],
        "model__kernel": ['linear', 'rbf', 'poly'],
        "model__gamma": ['scale', 'auto']
    },
    "Naive Bayes": {
    },
    "Decision Tree": {
        "model__max_depth": [10, 50, 100],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4]
    },
    "Random Forest": {
        "model__n_estimators": [100, 200, 300, 500],
        "model__max_depth": [5, 8, 10, 12],
        "model__min_samples_split": [2, 3, 4],
        "model__min_samples_leaf": [1, 2, 3]
    },
    "XGBoost": {
        "model__n_estimators": [100, 150, 200],
        "model__max_depth": [3, 5, 7],
        "model__learning_rate": [0.01, 0.1],
        "model__subsample": [0.8, 1.0]
    }
}

for name, model in baseline_models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=params_grid[name],
        scoring='f1',
        cv=5,
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)

    tuned_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "Best Params": grid_search.best_params_
    })

tuned_df = pd.DataFrame(tuned_results)

In [ ]:
tuned_df

,Model,Accuracy,Precision,Recall,F1,Best Params
0,Logistic Regression,0.819672,0.781250,0.862069,0.819672,{'model__C': 1}
1,KNN,0.868852,0.838710,0.896552,0.866667,"{'model__n_neighbors': 5, 'model__weights': 'u..."
2,Naive Bayes,0.852459,0.833333,0.862069,0.847458,{}
3,SVM,0.868852,0.862069,0.862069,0.862069,"{'model__C': 0.01, 'model__gamma': 'scale', 'm..."
4,Decision Tree,0.704918,0.677419,0.724138,0.700000,"{'model__max_depth': 10, 'model__min_samples_l..."
5,Random Forest,0.852459,0.857143,0.827586,0.842105,"{'model__max_depth': 5, 'model__min_samples_le..."
6,XGBoost,0.819672,0.800000,0.827586,0.813559,"{'model__learning_rate': 0.01, 'model__max_dep..."


# Observations

## Baseline Evaluation

All classification models were initially evaluated using 5-fold Cross Validation to obtain reliable performance estimates before hyperparameter optimization.

## Hyperparameter Tuning

Hyperparameter optimization was performed using GridSearchCV with 5-fold Cross Validation, ensuring a consistent evaluation strategy while searching for the best model parameters.

## Performance Comparison
- Hyperparameter tuning significantly improved the performance of KNN, SVM, Naive Bayes, and Random Forest, particularly in terms of accuracy and recall.
- KNN and SVM achieved the highest accuracy of 86.89%, outperforming the remaining classifiers.
- KNN achieved the highest recall (89.66%), making it the best model at identifying patients with heart disease, while SVM achieved a strong balance across all evaluation metrics, with 86.89% accuracy, 86.21% precision, 86.21% recall, and 86.21% F1-score.
- Logistic Regression and Decision Tree showed a slight decrease in performance after hyperparameter tuning, indicating that the default configurations were already close to optimal for this dataset.
- Decision Tree remained the weakest-performing model, recording the lowest accuracy and F1-score among all classifiers.

## Final Model Selection

Although KNN achieved the highest recall (89.66%), Support Vector Machine (SVM) is selected dur to the following reasons:

- SVM achieved the highest overall accuracy (86.89%), tied with KNN.
- It generally provides better generalization and is less sensitive to noisy data than KNN
- KNN performance can be affected by irrelevant features, data distribution, and the curse of dimensionality